# Preprocessing script for aggregating Gapminder datasets

In [14]:
from pathlib import Path

import pandas as pd


def melt_wide(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
	
	# Get country code and country name columns
	id_vars = ["geo", "name"]
	
	# Get year columns
	year_cols = [col for col in df.columns if str(col).isdigit()]
	
    # Melt the DataFrame from wide to long format
	long_df = df.melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name=value_name)
	long_df["year"] = long_df["year"].astype(int)
	
	return long_df

In [15]:
# Load the three datasets
data_path = Path("data")
gdp = pd.read_csv(data_path / "gdp_pcap.csv")
lex = pd.read_csv(data_path / "lex.csv")
pop = pd.read_csv(data_path / "pop.csv")

In [16]:
gdp.head()

,geo,name,1800,1801,1802,1803,1804,1805,1806,1807,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,afg,Afghanistan,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,...,9397.79550,9628.20479,9864.18612,10105.83025,10353.22669,10606.46351,10865.62717,11130.80231,11402.07162,11679.51560
1,ago,Angola,435.23259,436.64111,438.75389,440.86667,442.27519,444.38797,446.50074,448.61352,...,30471.68022,30991.50308,31513.98698,32038.95027,32566.20896,33095.57701,33626.86659,34159.88835,34694.45172,35230.36514
2,alb,Albania,547.53369,549.10393,550.67868,552.25795,553.84175,555.43009,557.02298,558.62044,...,57444.50359,57884.32605,58319.28840,58749.34444,59174.45223,59594.57402,60009.67615,60419.72899,60824.70684,61224.58783
3,and,Andorra,1598.53128,1601.20217,1603.87307,1607.87941,1610.55031,1613.22120,1615.89210,1618.56300,...,82535.68235,82627.86812,82718.08635,82806.37556,82892.77367,82977.31800,83060.04528,83140.99165,83220.19266,83297.68327
4,are,UAE,1332.77712,1336.78346,1342.12526,1347.46705,1352.80884,1356.81518,1362.15698,1367.49877,...,81900.09821,81885.61721,81871.47395,81857.66048,81844.16902,81830.99199,81818.12198,81805.55176,81793.27428,81781.28265


In [17]:
lex.head()

,geo,name,1800,1801,1802,1803,1804,1805,1806,1807,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,afg,Afghanistan,28.21,28.20,28.19,28.18,28.17,28.16,28.15,28.14,...,79.09,79.19,79.29,79.50,79.61,79.71,79.81,80.02,80.12,80.23
1,ago,Angola,26.98,26.98,26.98,26.98,26.98,26.98,26.98,26.98,...,73.56,73.66,73.76,73.95,74.05,74.15,74.35,74.45,74.55,74.65
2,alb,Albania,35.40,35.40,35.40,35.40,35.40,35.40,35.40,35.40,...,89.10,89.20,89.30,89.40,89.50,89.60,89.70,89.80,89.90,90.00
3,and,Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,91.89,92.09,92.19,92.29,92.39,92.49,92.59,92.69,92.79,92.89
4,are,UAE,30.70,30.70,30.70,30.70,30.70,30.70,30.70,30.70,...,87.64,87.73,87.83,87.93,88.02,88.12,88.22,88.31,88.41,88.51


In [18]:
pop.head()

,geo,name,1800,1801,1802,1803,1804,1805,1806,1807,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,afg,Afghanistan,3280000,3280000,3280000,3280000,3280000,3280000,3280000,3280000,...,124110001.0,124854162.0,125589411.0,126306889.0,126990147.0,127647762.0,128305496.0,128964762.0,129616118.0,130216739.0
1,ago,Angola,1567028,1567028,1567028,1567028,1567028,1567028,1567028,1567028,...,138919849.0,140249378.0,141550054.0,142835147.0,144102524.0,145340832.0,146549586.0,147743369.0,148912810.0,150045574.0
2,alb,Albania,400000,401773,403554,405343,407140,408945,410758,412579,...,1344055.0,1324192.0,1304876.0,1286145.0,1268008.0,1250429.0,1233362.0,1216785.0,1200681.0,1184997.0
3,and,Andorra,2654,2654,2654,2654,2654,2654,2654,2654,...,52781.0,52134.0,51484.0,50843.0,50211.0,49586.0,48973.0,48373.0,47791.0,47222.0
4,are,UAE,40153,40153,40153,40153,40153,40153,40153,40153,...,24075187.0,24298127.0,24521988.0,24747048.0,24973506.0,25201603.0,25431504.0,25663508.0,25897696.0,26134299.0


In [19]:
# Melt the datasets from wide to long format
gdp_long = melt_wide(gdp, "gdp_pcap")
lex_long = melt_wide(lex, "lex")
pop_long = melt_wide(pop, "pop")

# Merge the datasets on country code, country name, and year
merged = gdp_long.merge(lex_long, on=["geo", "name", "year"], how="outer")
merged = merged.merge(pop_long, on=["geo", "name", "year"], how="outer")

# Rename columns and reorder
merged = merged[["geo", "name", "year", "gdp_pcap", "lex", "pop"]]
merged = merged.sort_values(["geo", "year"], ignore_index=True)
merged.head()

,geo,name,year,gdp_pcap,lex,pop
0,afg,Afghanistan,1800,560.88817,28.21,3280000.0
1,afg,Afghanistan,1801,560.88817,28.20,3280000.0
2,afg,Afghanistan,1802,560.88817,28.19,3280000.0
3,afg,Afghanistan,1803,560.88817,28.18,3280000.0
4,afg,Afghanistan,1804,560.88817,28.17,3280000.0


In [20]:
# Save the merged dataset to a new CSV file
output_path = data_path / "gapminder_aggregated.csv"
merged.to_csv(output_path, index=False)